In [3]:
import os
import h5py as h5
import numpy as np
import xarray as xr
from scipy.sparse import csr_array
import pandas as pd
import matplotlib.pyplot as plt
import rioxarray

In [4]:
prefix, sz_zone = 'sz', '_hikkerk'
h5_file = f"discretised{sz_zone}/{prefix}_tsunami_gf_dict.h5"
tif_dir = os.path.join("Z:\\", "McGrath", "TsunamiGrids", "NSHMvsPTHM", "Hikkerm")
if 'hik' in sz_zone:
    rupt_dir = os.path.join('..', 'data', 'sz_solutions', 'NZSHM22_ScaledInversionSolution-QXV0b21hdGlvblRhc2s6MTA3NzEx') # Puysegur
else:
    rupt_dir = os.path.join('..', 'data', 'sz_solutions', 'NZSHM22_ScaledInversionSolution-QXV0b21hdGlvblRhc2s6MTMyNzM5NQ==') # Puysegur

rupt_inds = os.path.join(rupt_dir, 'ruptures', 'indices.csv')
rupt_slip = os.path.join(rupt_dir, 'ruptures', 'average_slips.csv')
rupt_mags = os.path.join(rupt_dir, 'ruptures', 'properties.csv')
rupt_rates = os.path.join(rupt_dir, 'solution', 'rates.csv')

min_rate = 0
min_mag = 6
max_mag = 8

to_grd = True

rake90 = False
if rake90:
    h5_file = h5_file.replace(".h5", "_rake90.h5")

gf_h5 = h5.File(h5_file, 'r')

minx, miny, maxx, maxy, res = gf_h5['grid_extent'][:]
height = int(np.ceil((maxy - miny) / res) + 1)
width = int(np.ceil((maxx - minx) / res) + 1)

lons, lats = np.arange(minx, maxx + res, res), np.arange(miny, maxy + res, res)

In [5]:
rates = pd.read_csv(rupt_rates, index_col=0)
rates = rates[rates['Annual Rate'] > min_rate]
mags = pd.read_csv(rupt_mags)
rates['Magnitude'] = pd.read_csv(rupt_mags).loc[rates.index, 'Magnitude']
rates = rates[rates['Magnitude'] >= min_mag]
rates = rates[rates['Magnitude'] < max_mag]
rates['Slip'] = pd.read_csv(rupt_slip, index_col=0).loc[rates.index, 'Average Slip (m)']
rates = rates.sort_values(by='Magnitude', ascending=False)
patch_indices = pd.read_csv(rupt_inds, index_col=0).loc[rates.index]
print(f"n_ruptures: {len(rates)}")

n_ruptures: 295


In [6]:
for rupt_id, (rate, magnitude, slip) in rates.iterrows():
    print(f"Processing rupture ID: {rupt_id}, Mag: {magnitude:.2f}, Slip: {slip:.2f} m, Rate: {rate:.2e}")
    # slip = rates.loc[rupt_id, 'Slip']
    rupture = patch_indices.loc[rupt_id]
    n_sections = int(rupture['Num Sections'])
    indices = [str(int(section)) for section in rupture[1:n_sections+1].values]
    grid = csr_array(np.zeros((height, width)))
    for section in indices:
        section_gf = gf_h5[section]
        vert_data = section_gf['vertical'][:]
        vert_indices = section_gf['vertical_indices'][:]
        vert_indptr = section_gf['vertical_indptr'][:]
        vert_shape = section_gf['shape'][:]
        section_grid = csr_array((vert_data, vert_indices, vert_indptr), shape=vert_shape)
        grid += section_grid * slip
    
    if grid.data.shape[0] == 0:
        continue
    # plt.imshow(grid.toarray(), extent=(minx, maxx, miny, maxy), vmin=-max(np.abs(grid.data)), vmax=max(np.abs(grid.data)), cmap='RdBu_r', origin='lower'), plt.colorbar()
    # plt.title(f'Rupture ID: {rupt_id}, Mag: {magnitude:.2f}, Slip: {slip:.2f} m')
    # plt.show()

    # da = (
    # xr.DataArray(
    #     grid.toarray().astype(np.float32),
    #     dims=("lat", "lon"),
    #     coords={"lon": lons, "lat": lats},
    # )
    # .rio.set_spatial_dims("lon", "lat")
    # .rio.write_crs("EPSG:2193")
    # )

    # da.rio.to_raster(os.path.join(tif_dir, f"rupt_{rupt_id}_Mw{magnitude:.2f}{'_rake90' if rake90 else ''}.tif"))


    start_col, end_col = grid.indices.min(), grid.indices.max()
    start_col = start_col if start_col == 0 else start_col - 1
    end_col = end_col if end_col == grid.shape[1] - 1 else end_col + 1
    start_row = np.where(grid.indptr > 0)[0][0]
    end_row = np.where(grid.indptr == grid.data.shape[0])[0][0]
    start_row = start_row if start_row == 0 else start_row - 1
    end_row = end_row if end_row == grid.indptr.shape[0] - 1 else end_row + 1

    new_data = grid.data
    new_indices = grid.indices - start_col
    new_indptr = grid.indptr[start_row-1:end_row+1]
    new_sparse = csr_array((new_data, new_indices, new_indptr), shape=(end_row - start_row + 1, end_col - start_col + 1))

    da = (
    xr.DataArray(
        new_sparse.toarray().astype(np.float32),
        dims=("lat", "lon"),
        coords={"lon": lons[start_col:end_col+1], "lat": lats[start_row-1:end_row]},
    )
    .rio.set_spatial_dims("lon", "lat")
    .rio.write_crs("EPSG:2193")
    )

    outfile = os.path.join(tif_dir, f"rupt_{rupt_id}_Mw{magnitude:.2f}{'_rake90' if rake90 else ''}")
    if to_grd:
        da.to_netcdf(f"{outfile}.grd")
    else:
        da.rio.to_raster(f"{outfile}.tif")

gf_h5.close()
print("Done :)")
print(os.path.join(tif_dir, f"rupt_{'*'}_Mw{'*'}{'_rake90' if rake90 else ''}.{'grd' if to_grd else 'tif'}"))

Processing rupture ID: 12905, Mag: 7.99, Slip: 3.72 m, Rate: 6.83e-06
Processing rupture ID: 13542, Mag: 7.99, Slip: 3.72 m, Rate: 3.06e-05
Processing rupture ID: 13855, Mag: 7.99, Slip: 3.72 m, Rate: 4.45e-05
Processing rupture ID: 15641, Mag: 7.99, Slip: 3.72 m, Rate: 9.69e-06
Processing rupture ID: 20879, Mag: 7.99, Slip: 3.71 m, Rate: 1.02e-04
Processing rupture ID: 21436, Mag: 7.99, Slip: 3.71 m, Rate: 6.86e-05
Processing rupture ID: 21793, Mag: 7.99, Slip: 3.71 m, Rate: 3.22e-05
Processing rupture ID: 22135, Mag: 7.99, Slip: 3.71 m, Rate: 7.71e-05
Processing rupture ID: 22456, Mag: 7.99, Slip: 3.70 m, Rate: 8.68e-05
Processing rupture ID: 23011, Mag: 7.99, Slip: 3.70 m, Rate: 4.35e-05
Processing rupture ID: 23229, Mag: 7.99, Slip: 3.70 m, Rate: 2.61e-04
Processing rupture ID: 23409, Mag: 7.99, Slip: 3.70 m, Rate: 5.54e-05
Processing rupture ID: 23481, Mag: 7.99, Slip: 3.70 m, Rate: 2.78e-04
Processing rupture ID: 23616, Mag: 7.99, Slip: 3.70 m, Rate: 1.32e-03
Processing rupture I